# Retrieval Showcase (Using Existing Evaluation Dataset)

This notebook demonstrates retrieval quality using the already built evaluation assets in `backend/evaluation`.

What it shows:
- sample questions from `backend/evaluation/dataset/qa_pairs.json`
- retrieval with the same section-scoped logic used in `evaluate_retrieval.py`
- compact metrics and top-hit previews for presentation

In [2]:
from __future__ import annotations

import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'playground' else Path.cwd()
BACKEND_DIR = ROOT / 'backend'
for p in (ROOT, BACKEND_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

DATASET_PATH = BACKEND_DIR / 'evaluation' / 'dataset' / 'qa_pairs.json'
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}')

print('ROOT:', ROOT)
print('BACKEND_DIR:', BACKEND_DIR)
print('DATASET_PATH:', DATASET_PATH)

ROOT: /home/aman/storage/Python/Projects/Research Paper Assistant
BACKEND_DIR: /home/aman/storage/Python/Projects/Research Paper Assistant/backend
DATASET_PATH: /home/aman/storage/Python/Projects/Research Paper Assistant/backend/evaluation/dataset/qa_pairs.json


In [2]:
from collections import defaultdict
from rag.retrieval import RetrievalPipeline
from evaluation.evaluate_retrieval import (
    _load_dataset,
    _compute_mrr,
    _precision_at_k,
    _recall_at_k,
    )

TOP_K = 5
TOP_N = 5
MAX_QUESTIONS = 12  # keep demo quick while still grounded in the real dataset

entries = _load_dataset(DATASET_PATH)
print('Valid evaluation questions:', len(entries))

# Balanced subset for presentation: up to 3 per paper_id.
by_paper = defaultdict(list)
for e in entries:
    by_paper[e.get('paper_id', 'unknown')].append(e)

demo_entries = []
for pid in sorted(by_paper.keys()):
    demo_entries.extend(by_paper[pid][:3])

demo_entries = demo_entries[:MAX_QUESTIONS]
print('Questions used in this demo:', len(demo_entries))

pipeline = RetrievalPipeline()

rows = []
preview_rows = []
for entry in demo_entries:
    question = str(entry.get('question', '') or '')
    section_id = str(entry.get('section_id', '') or '')
    document_id = str(entry.get('document_id', '') or '')
    relevant_ids = [str(x) for x in (entry.get('relevant_chunk_ids') or [])]
    relevant_set = set(relevant_ids)

    hits = pipeline.retrieve_with_section_scope(
        query=question,
        section_id=section_id,
        document_id=document_id,
        top_k=TOP_K,
        top_n=TOP_N,
        rerank=True,
    )

    retrieved_ids = []
    for h in hits:
        meta = getattr(h, 'metadata', {}) or {}
        cid = meta.get('_id')
        if cid:
            retrieved_ids.append(str(cid))

    p2 = _precision_at_k(retrieved_ids, relevant_set, 2)
    p5 = _precision_at_k(retrieved_ids, relevant_set, 5)
    r3 = _recall_at_k(retrieved_ids, relevant_set, 3)
    r5 = _recall_at_k(retrieved_ids, relevant_set, 5)
    rr = _compute_mrr(retrieved_ids[:5], relevant_set)

    rows.append({
        'paper_id': entry.get('paper_id', ''),
        'paper_type': entry.get('paper_type', ''),
        'question_type': entry.get('question_type', ''),
        'question': question,
        'precision_at_2': round(p2, 3),
        'precision_at_5': round(p5, 3),
        'recall_at_3': round(r3, 3),
        'recall_at_5': round(r5, 3),
        'reciprocal_rank': round(rr, 3),
        'relevant_count': len(relevant_set),
        'retrieved_count': len(retrieved_ids),
    })

    top = hits[0] if hits else None
    top_meta = (getattr(top, 'metadata', {}) or {}) if top else {}
    top_text = ' '.join(str(getattr(top, 'content', '') or '').split()) if top else ''
    preview_rows.append({
        'paper_id': entry.get('paper_id', ''),
        'question': question,
        'top_score': round(float(getattr(top, 'score', 0.0) or 0.0), 4) if top else 0.0,
        'top_section_title': str(top_meta.get('section_title') or ''),
        'top_chunk_id': str(top_meta.get('_id') or ''),
        'top_preview': top_text[:240] + ('...' if len(top_text) > 240 else ''),
    })

metrics_df = pd.DataFrame(rows)
preview_df = pd.DataFrame(preview_rows)

display(metrics_df.head(20))
display(preview_df.head(10))

Configuration loaded:
  - API Host: 0.0.0.0:8000
  - Upload Directory: uploads
  - Max File Size: 50 MB
  - Groq API Configured: True
  - Qdrant Configured: True
  - Qdrant Collection: research_papers
  - LangSmith Tracing: Enabled
Valid evaluation questions: 32
Questions used in this demo: 12


/home/aman/storage/Python/Projects/Research Paper Assistant/env_research/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:rag.retrieval.search.reranker:FlashRankReranker: loaded model ms-marco-MiniLM-L-12-v2
INFO:rag.retrieval.search.reranker:FlashRankReranker: reranked 5 → 5 results for query 'What BLEU scores did the Transformer achieve on the WMT 2014'
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/p

,paper_id,paper_type,question_type,question,precision_at_2,precision_at_5,recall_at_3,recall_at_5,reciprocal_rank,relevant_count,retrieved_count
0,paper_applied,Applied,factual,What BLEU scores did the Transformer achieve o...,0.5,0.2,1.0,1.0,1.000,1,5
1,paper_applied,Applied,factual,How does the training cost of the Transformer ...,0.5,0.2,1.0,1.0,1.000,1,5
2,paper_applied,Applied,factual,What effect does reducing the attention key si...,1.0,0.4,1.0,1.0,1.000,2,5
3,paper_memgpt,Applied,factual,What are the two long-context domains in which...,1.0,0.4,1.0,1.0,1.000,2,5
4,paper_memgpt,Applied,factual,What accuracy improvement does adding MemGPT p...,0.5,0.2,1.0,1.0,1.000,1,5
5,paper_memgpt,Applied,conceptual,What is the nested key-value retrieval task an...,0.5,0.4,1.0,1.0,1.000,2,5
6,paper_survey,Survey,factual,What are the three advantages of pre-training ...,0.5,0.4,1.0,1.0,0.500,2,5
7,paper_survey,Survey,factual,What two shallow architectures did Mikolov et ...,0.0,0.2,1.0,1.0,0.333,1,5
8,paper_survey,Survey,factual,What was the first successful PTM for NLP and ...,0.5,0.2,1.0,1.0,1.000,1,5
9,paper_theory,Theory,factual,What is the upper bound on the query complexit...,1.0,0.4,1.0,1.0,1.000,2,5


,paper_id,question,top_score,top_section_title,top_chunk_id,top_preview
0,paper_applied,What BLEU scores did the Transformer achieve o...,0.9999,,5cba0513-5e97-5002-bf12-636818292019,The Transformer model outperforms previous sta...
1,paper_applied,How does the training cost of the Transformer ...,0.9981,,5cba0513-5e97-5002-bf12-636818292019,The Transformer model outperforms previous sta...
2,paper_applied,What effect does reducing the attention key si...,0.9918,,e5074e88-6051-5b75-91cd-96e06c4288be,We used beam search as described in the previo...
3,paper_memgpt,What are the two long-context domains in which...,0.9998,,36e429aa-83d5-5fdb-b202-e08fb140a787,We assess MemGPT in two long-context domains: ...
4,paper_memgpt,What accuracy improvement does adding MemGPT p...,1.0000,,8715edec-87f3-509e-9628-739023818ca0,The performance of the Deep Memory Retrieval (...
5,paper_memgpt,What is the nested key-value retrieval task an...,0.9916,,36e429aa-83d5-5fdb-b202-e08fb140a787,We assess MemGPT in two long-context domains: ...
6,paper_survey,What are the three advantages of pre-training ...,0.9921,,d9abed9d-265e-53fa-bdb4-63cf7c7e1580,"In contrast, large-scale unlabeled corpora are..."
7,paper_survey,What two shallow architectures did Mikolov et ...,0.9996,,536d2901-5b41-5163-8345-daec23b3821b,Mikolov et al. [11] showed that there is no ne...
8,paper_survey,What was the first successful PTM for NLP and ...,0.9974,,b80adbde-760d-5441-a1c3-04056103c010,Dai and Le [35] proposed the ﬁrst successful i...
9,paper_theory,What is the upper bound on the query complexit...,0.9066,,f87fdaf6-6569-5ec1-8c3c-baaa5b533e01,"5.2.3 Irreducible representations πλ(g)i,j Pro..."


In [3]:
if metrics_df.empty:
    print('No metrics available.')
else:
    overall = metrics_df[[
        'precision_at_2', 'precision_at_5', 'recall_at_3', 'recall_at_5', 'reciprocal_rank'
    ]].mean().round(3).to_frame('value')
    print('Overall metrics (demo subset):')
    display(overall)

    by_paper_type = metrics_df.groupby('paper_type', as_index=False)[[
        'precision_at_2', 'precision_at_5', 'recall_at_3', 'recall_at_5', 'reciprocal_rank'
    ]].mean().round(3)
    print('By paper type:')
    display(by_paper_type)

    by_question_type = metrics_df.groupby('question_type', as_index=False)[[
        'precision_at_2', 'precision_at_5', 'recall_at_3', 'recall_at_5', 'reciprocal_rank'
    ]].mean().round(3)
    print('By question type:')
    display(by_question_type)

    print('Lowest-scoring questions (helpful for discussion):')
    display(
        metrics_df.sort_values(['precision_at_5', 'reciprocal_rank']).head(5)[[
            'paper_id', 'question_type', 'question', 'precision_at_5', 'reciprocal_rank'
        ]]
    )

Overall metrics (demo subset):


,value
precision_at_2,0.583
precision_at_5,0.300
recall_at_3,0.958
recall_at_5,0.958
reciprocal_rank,0.819


By paper type:


,paper_type,precision_at_2,precision_at_5,recall_at_3,recall_at_5,reciprocal_rank
0,Applied,0.667,0.300,1.000,1.000,1.000
1,Survey,0.333,0.267,1.000,1.000,0.611
2,Theory,0.667,0.333,0.833,0.833,0.667


By question type:


,question_type,precision_at_2,precision_at_5,recall_at_3,recall_at_5,reciprocal_rank
0,conceptual,0.500,0.333,0.833,0.833,0.667
1,factual,0.611,0.289,1.000,1.000,0.870


Lowest-scoring questions (helpful for discussion):


,paper_id,question_type,question,precision_at_5,reciprocal_rank
7,paper_survey,factual,What two shallow architectures did Mikolov et ...,0.2,0.333
11,paper_theory,conceptual,What is the role of the partial transpose oper...,0.2,0.500
0,paper_applied,factual,What BLEU scores did the Transformer achieve o...,0.2,1.000
1,paper_applied,factual,How does the training cost of the Transformer ...,0.2,1.000
4,paper_memgpt,factual,What accuracy improvement does adding MemGPT p...,0.2,1.000


In [3]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'playground' else Path.cwd()
BACKEND_DIR = ROOT / 'backend'
RESULTS_DIR = BACKEND_DIR / 'evaluation' / 'results'

with open(RESULTS_DIR / 'ablation_results.json') as f:
    orig = json.load(f)

with open(RESULTS_DIR / 'new_ablation_results.json') as f:
    new = json.load(f)

display_names = [
    "Baseline (dense only)",
    "Hybrid BM25 + dense (no reranker)",
    "Full system (hybrid + reranker + section filter)",
]

def make_df(configs):
    rows = []
    for cfg, name in zip(configs, display_names):
        rows.append({
            'Configuration': name,
            'P@3':  cfg['precision_at_3'],
            'P@5':  cfg['precision_at_5'],
            'R@5':  cfg['recall_at_5'],
            'MRR':  cfg['reciprocal_rank'],
        })
    return pd.DataFrame(rows).set_index('Configuration')

orig_configs = orig['summary']['configurations']
new_configs  = new['summary']['configurations']

orig_n = orig_configs[0]['num_questions']
new_n  = new_configs[0]['num_questions']

print(f"Ablation Study — {orig_n}-question single-section dataset")
print("=" * 70)
display(make_df(orig_configs))

print(f"\nAblation Study — {new_n}-question cross-section dataset")
print("=" * 70)
display(make_df(new_configs))

Ablation Study — 32-question single-section dataset


,P@3,P@5,R@5,MRR
Configuration,,,,
Baseline (dense only),0.25,0.182,0.651,0.381
Hybrid BM25 + dense (no reranker),0.31,0.261,0.703,0.487
Full system (hybrid + reranker + section filter),0.45,0.330,0.938,0.590



Ablation Study — 40-question cross-section dataset


,P@3,P@5,R@5,MRR
Configuration,,,,
Baseline (dense only),0.48,0.421,0.531,0.431
Hybrid BM25 + dense (no reranker),0.51,0.452,0.572,0.531
Full system (hybrid + reranker + section filter),0.69,0.630,0.761,0.682


## Ablation Study (Existing Evaluation Logic)

This section reuses the already built ablation evaluation code from `backend/evaluation/evaluate_ablation.py` and runs it on a presentation-sized subset.

In [5]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'playground' else Path.cwd()
BACKEND_DIR = ROOT / 'backend'
RESULTS_DIR = BACKEND_DIR / 'evaluation' / 'results'

with open(RESULTS_DIR / 'answer_results.json') as f:
    orig = json.load(f)

with open(RESULTS_DIR / 'new_answer_results.json') as f:
    new = json.load(f)

def make_answer_df(data, label):
    m = data['metrics']
    return pd.DataFrame([{
        'Dataset': label,
        'Faithfulness':      m['faithfulness'],
        'Answer Relevancy':  m['answer_relevancy'],
        'Context Precision': m['context_precision'],
    }]).set_index('Dataset')

df = pd.concat([
    make_answer_df(orig, 'Single-section (32Q)'),
    make_answer_df(new,  'Cross-section (40Q)'),
])

print("Answer Quality Metrics")
print("=" * 55)
display(df.round(3))

Answer Quality Metrics


,Faithfulness,Answer Relevancy,Context Precision
Dataset,,,
Single-section (32Q),0.835,0.886,0.571
Cross-section (40Q),0.789,0.829,0.742
